### Q1: Embedding a Query

Embed the following query:

```bash
How does approximate nearest neighbor search work?
```

The embedder returns a vector of 384 numbers. What's the first value (v[0])?

In [1]:
from embedder import Embedder

embedder = Embedder()
v = embedder.encode("How does approximate nearest neighbor search work?")
print(v[0])

2026-06-28 11:13:02.130648505 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


-0.02058203437252893


A1: -0.02

### Q2. Cosine similarity
The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?

In [2]:
from gitsource import GithubRepositoryDataReader
import numpy as np

# Load documents
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

# Get the specific file
target = [d for d in documents if d['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md'][0]

# Embed both and compute dot product (= cosine similarity since vectors are normalized)
embedder = Embedder()
v = embedder.encode("How does approximate nearest neighbor search work?")
v_doc = embedder.encode(target['content'])

similarity = np.dot(v, v_doc)
print(similarity)

0.36107027225589694


A2: 0.36

### Q3. Chunking and search by hand
A full page covers several topics, which waters down its embedding.

We chunk the pages the same way as in homework 1:

```bash
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
```
We embed every chunk's content with encode_batch, stack the vectors into a matrix X, and score the Q1 query against all chunks:

```bash
scores = X.dot(v)
```
Which file does the highest-scoring chunk belong to (its filename)?

In [3]:
from gitsource import chunk_documents

# Chunk the documents
chunks = chunk_documents(documents, size=2000, step=1000)

# Embed all chunks
contents = [chunk['content'] for chunk in chunks]
X = embedder.encode_batch(contents)

# Embed the query
v = embedder.encode("How does approximate nearest neighbor search work?")

# Score all chunks
scores = X.dot(v)

# Find the best chunk
best_idx = np.argmax(scores)
print(scores[best_idx])
print(chunks[best_idx]['filename'])

0.6489017718578813
02-vector-search/lessons/07-sqlitesearch-vector.md


A3: 02-vector-search/lessons/07-sqlitesearch-vector.md

### Q4. Vector search with minsearch
We've done vector search by hand, which is good for learning, but it's not what we do in practice. In practice we use libraries.

Let's use VectorSearch from minsearch and run a search for the following query:

What metric do we use to evaluate a search engine?

Which file is the filename of the first result?

In [4]:
from minsearch import VectorSearch

# Build the index
index = VectorSearch(keyword_fields=['filename'])
index.fit(embedder.encode_batch([c['content'] for c in chunks]), chunks)

# Search
query = "What metric do we use to evaluate a search engine?"
v_query = embedder.encode(query)
results = index.search(v_query, num_results=5)

print(results[0]['filename'])

04-evaluation/lessons/05-search-metrics.md


A4: 04-evaluation/lessons/05-search-metrics.md

### Q5. Text search vs vector search
Vector search matches by meaning, keyword search by exact words.

Let's compare them. Index the same chunks with Index from minsearch. Use content as a text field.

Run both searches for this query:

```bash
How do I store vectors in PostgreSQL?
```

Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?

In [5]:
from minsearch import Index

# Keyword search index
text_index = Index(text_fields=['content'], keyword_fields=['filename'])
text_index.fit(chunks)

query = "How do I store vectors in PostgreSQL?"

# Vector search top 5
v_query = embedder.encode(query)
vector_results = index.search(v_query, num_results=5)

# Text search top 5
text_results = text_index.search(query, num_results=5)

# Compare filenames
vector_files = set(r['filename'] for r in vector_results)
text_files = set(r['filename'] for r in text_results)

print("In vector but not text:", vector_files - text_files)

In vector but not text: {'02-vector-search/lessons/08-pgvector.md'}


A5: 02-vector-search/lessons/08-pgvector.md

### Q6: Hybrid Search

In [6]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [7]:
query = "How do I give the model access to tools?"

v_query = embedder.encode(query)
vector_results = index.search(v_query, num_results=5)
text_results = text_index.search(query, num_results=5)

results = rrf([vector_results, text_results])
print(results[0]['filename'])

01-agentic-rag/lessons/13-function-calling.md


A6: 01-agentic-rag/lessons/13-function-calling.md